# Noisy 4-to-1 MUX: QFOIL explanations

This notebook accompanies the automated tests with examples written out by hand. The feature order is `S0, S1, D0, D1, D2, D3, N0, N1`. The MUX uses `index = 2*S1 + S0`; `N0` and `N1` are independent noise features and never affect the output.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from qfoil import QFOIL
from qfoil.backends.pyjuice import PyJuiceProbabilityBackend
from examples.models.mux import MuxDistribution, build_mux_model

distribution = MuxDistribution.bernoulli()
model = build_mux_model(distribution, device="cuda:0")

Bernoulli distribution probabilities
- P(S0=1) = 0.35
- P(S1=1) = 0.65
- P(D=1) = [0.10, 0.30, 0.70, 0.90]
- P(N0=1) = 0.20
- P(N1=1) = 0.80

In [2]:
parser = QFOIL(
    probabilistic_backend=PyJuiceProbabilityBackend(model),
    instance_len=8,
    sigma_search_budget=0,
    sigma_mode="restricted",
)
evaluator = parser.evaluator

## Evidence used in this walkthrough

We observe `S0=0`, `S1=1`, and `D2=1`. The selectors choose `D2`, so this evidence determines that the class is 1.

In [4]:
evidence = (0, 1, None, None, 1, None, None, None)
class_id = 1

# Show the evidence and its exact ground-truth probability.
evidence, distribution.evidence_probability(evidence)

((0, 1, None, None, 1, None, None, None), 0.29575000000000007)

## VAL

`VAL(x)` represents $P(x)$, and `VAL(x,k)` represents $P(x,Y=k)$.

In [5]:
val = parser.parse("VAL(x)")
val_for_class = parser.parse("VAL(x, k)")

qfoil_val = evaluator.evaluate(val, {"x": evidence})
qfoil_joint = evaluator.evaluate(
    val_for_class, {"x": evidence, "k": class_id}
)

# Show QFOIL and ground truth side by side. They must agree.
print("VAL:", qfoil_val, distribution.evidence_probability(evidence))
print(
    "VAL for the class:",
    qfoil_joint,
    distribution.evidence_probability(evidence, class_id),
)

VAL: 0.2957500113001942 0.29575000000000007
VAL for the class: 0.2957500113001942 0.29575000000000007


## EP y CEP

`EP(x,k)` is $P(Y=k\mid x)$. `CEP(x,k)` is its complement $P(Y\neq k\mid x)$.

In [6]:
ep = parser.parse("EP(x, k)")
cep = parser.parse("CEP(x, k)")

qfoil_ep = evaluator.evaluate(ep, {"x": evidence, "k": class_id})
qfoil_cep = evaluator.evaluate(cep, {"x": evidence, "k": class_id})
ground_truth_ep = distribution.posterior_class(evidence, class_id)

# Show EP, ground truth, CEP, and EP+CEP. We expect 1, 1, 0, and 1.
qfoil_ep, ground_truth_ep, qfoil_cep, qfoil_ep + qfoil_cep

(1.0, 1.0, 0.0, 1.0)

## Noise invariance

Observing `N0` and `N1` changes the probability of the evidence, but it must not change the class posterior because both noise features are independent of the MUX output.

In [7]:
evidence_without_noise = (None, 1, None, None, None, None, None, None)
evidence_with_noise = (None, 1, None, None, None, None, 0, 1)

ep_without_noise = evaluator.evaluate(ep, {"x": evidence_without_noise, "k": class_id})
ep_with_noise = evaluator.evaluate(ep, {"x": evidence_with_noise, "k": class_id})
val_without_noise = evaluator.evaluate(val, {"x": evidence_without_noise})
val_with_noise = evaluator.evaluate(val, {"x": evidence_with_noise})

# EP values must match; VAL values should differ.
ep_without_noise, ep_with_noise, val_without_noise, val_with_noise

(0.770000004712191, 0.770000004712191, 0.6499999625119839, 0.4160000037491872)

## COND

`COND(event, given, k)` represents $P(event,Y=k\mid given)$. Here we condition on the selectors and ask for `D2=1` together with class 1.

In [8]:
event = (None, None, None, None, 1, None, None, None)
given = (0, 1, None, None, None, None, None, None)
cond = parser.parse("COND(event, given, k)")

qfoil_cond = evaluator.evaluate(
    cond, {"event": event, "given": given, "k": class_id}
)

fused = (0, 1, None, None, 1, None, None, None)
ground_truth_cond = (
    distribution.evidence_probability(fused, class_id)
    / distribution.evidence_probability(given)
)

# Show both values. They must agree.
qfoil_cond, ground_truth_cond

(0.6999999823196892, 0.6999999999999998)

## Uniform and Bernoulli distributions

This notebook currently uses `MuxDistribution.bernoulli()`. You can replace it with `MuxDistribution.uniform()` in the first cell and run the notebook again. The equalities still hold, but the probabilities become symmetric.

## Probable sufficient reasons (PSR)

A candidate `reason` is a PSR for a complete `instance` when it is contained in the instance and `EP(reason, class_id) >= delta`.

In [9]:
instance = (0, 1, 0, 1, 1, 0, 0, 1)
reason = (0, 1, None, None, 1, None, None, None)
class_id = 1
delta = 0.9

psr = parser.parse(
    "FULL(x) * (y <<< x) * (EP(y, k) >= delta)"
)
is_psr = evaluator.evaluate(
    psr,
    {"x": instance, "y": reason, "k": class_id, "delta": delta},
)

# Show the QFOIL result and the exact posterior. Expect True and 1.0.
bool(is_psr), distribution.posterior_class(reason, class_id)

(True, 1.0)

In [10]:
# this is a negative example, where the reason is not sufficient to explain the instance for the given delta.
not_reason = (None, 1, None, None, None, None, None, None)
is_psr = evaluator.evaluate(
    psr,
    {"x": instance, "y": not_reason, "k": class_id, "delta": delta},
)

bool(is_psr), distribution.posterior_class(not_reason, class_id), evaluator.evaluate(parser.parse("EP(y, k)"), {"y": not_reason, "k": class_id})

(False, 0.7699999999999998, 0.770000004712191)

## MinimumPSR

For `delta=0.9`, `(S1=1, D2=1)` is already sufficient. Removing either observed feature drops the posterior below the threshold. The `SIGMA` expression checks that no smaller sufficient reason exists.

In [11]:
minimum_reason = (None, 1, None, None, 1, None, None, None)
only_s1 = (None, 1, None, None, None, None, None, None)
only_d2 = (None, None, None, None, 1, None, None, None)

# Show the three posteriors. Only the two-feature reason exceeds 0.9.
print("S1 and D2:", distribution.posterior_class(minimum_reason, class_id))
print("Only S1:", distribution.posterior_class(only_s1, class_id))
print("Only D2:", distribution.posterior_class(only_d2, class_id))

S1 and D2: 0.9650000000000001
Only S1: 0.7699999999999998
Only D2: 0.6867500000000003


In [12]:
parser.define_function(
    "SE",
    parameters=["x", "candidate", "c", "delta", "size_max"],
    body=(
        "FULL(x) * (candidate <<< x) * "
        "(EP(candidate, c) >= delta) * "
        "(1 <= SIZE(candidate)) * "
        "(SIZE(candidate) <= size_max)"
    ),
)

minimum_psr_expression = (
    "SE(x, y, c, delta, size_max) * "
    "(SIGMA z. ("
    "    SE(x, z, c, delta, size_max) * "
    "    NOT(SIZE(y) <= SIZE(z))"
    ") == 0)"
)
minimum_psr = parser.parse(minimum_psr_expression)

is_minimum = evaluator.evaluate(
    minimum_psr,
    {
        "x": instance,
        "y": minimum_reason,
        "c": class_id,
        "delta": delta,
        "size_max": 8,
    },
)

# Show the predicate result. Expect True.
bool(is_minimum)

True

In [13]:
# Check that the only S1 reason is not a minimum sufficient reason.
bool(evaluator.evaluate(
    minimum_psr,
    {
        "x": instance,
        "y": only_s1,
        "c": class_id,
        "delta": delta,
        "size_max": 8,
    },
))

False

## Most probable sufficient reason (MPSR)

At `delta=0.65`, which is above the class prior `P(Y=1)=0.56`, this model has two one-feature minimum reasons: `S1=1` and `D2=1`. Neither noise feature qualifies because observing noise leaves the posterior equal to the prior. MPSR selects `D2=1` because it has the larger `VAL`.

In [14]:
reason_s1 = (None, 1, None, None, None, None, None, None)
reason_d2 = (None, None, None, None, 1, None, None, None)
noise_n0 = (None, None, None, None, None, None, 0, None)
noise_n1 = (None, None, None, None, None, None, None, 1)

val = parser.parse("VAL(x)")

# MPSR ranks P(reason) through VAL(reason), not P(reason, Y=k).
# Show VAL for the two minimum sufficient reasons. D2=1 is more probable.
print("S1=1:", evaluator.evaluate(val, {"x": reason_s1}))
print("D2=1:", evaluator.evaluate(val, {"x": reason_d2}))

S1=1: 0.6499999625119839
D2=1: 0.7000000031813146


In [15]:
print("S1=1:", distribution.evidence_probability(reason_s1))
print("D2=1:", distribution.evidence_probability(reason_d2))
print("EP for N0=0:", distribution.posterior_class(noise_n0, class_id))
print("EP for N1=1:", distribution.posterior_class(noise_n1, class_id))

S1=1: 0.6500000000000002
D2=1: 0.6999999999999998
EP for N0=0: 0.5600000000000002
EP for N1=1: 0.5599999999999999


In [16]:
parser.define_function(
    "MINIMUM_PSR",
    parameters=["x", "y", "c", "delta", "size_max"],
    body=(
        "SE(x, y, c, delta, size_max) * "
            "(SIGMA z. ("
            "    SE(x, z, c, delta, size_max) * "
            "    NOT(SIZE(y) <= SIZE(z))"
            ") == 0)"
    ),
)

In [17]:
most_probable_sr_expression = (
    "MINIMUM_PSR(x, y, c, delta, size_max) * "
    "(SIGMA z. ("
    "    MINIMUM_PSR(x, z, c, delta, size_max) * "
    "    NOT(VAL(z) <= VAL(y))"
    ") == 0)"
)

In [18]:
# Show that D2=1 is the most probable sufficient reason at delta=0.65.
most_probable_sr = parser.parse(most_probable_sr_expression)

bool(evaluator.evaluate(
    most_probable_sr,
    {
        "x": instance,
        "y": reason_d2,
        "c": class_id,
        "delta": 0.65,
        "size_max": 8,
    },
))

True

## Probable contrastive explanations (PCE)

A PCE releases features from a complete instance until the probability of a class different from `class_id` reaches `delta`. This probability is `CEP(candidate, class_id)`.

In [19]:
pce_instance = (0, 0, 1, 0, 0, 0, 0, 1)
pce_class_id = 1

release_s0 = (None, 0, 1, 0, 0, 0, 0, 1)
release_s1 = (0, None, 1, 0, 0, 0, 0, 1)
release_d0 = (0, 0, None, 0, 0, 0, 0, 1)
release_d1 = (0, 0, 1, None, 0, 0, 0, 1)
release_d0_and_n0 = (0, 0, None, 0, 0, 0, None, 1)

# Show the exact CEP after releasing one feature.
print("Release S0:", 1.0 - distribution.posterior_class(release_s0, pce_class_id))
print("Release S1:", 1.0 - distribution.posterior_class(release_s1, pce_class_id))
print("Release D0:", 1.0 - distribution.posterior_class(release_d0, pce_class_id))
print("Release D1:", 1.0 - distribution.posterior_class(release_d1, pce_class_id))

Release S0: 0.35
Release S1: 0.65
Release D0: 0.9
Release D1: 0.0


In [20]:
cep = parser.parse("CEP(x, k)")
print("CEP(release_s0, 1):", evaluator.evaluate(cep, {"x": release_s0, "k": pce_class_id}))
print("CEP(release_s1, 1):", evaluator.evaluate(cep, {"x": release_s1, "k": pce_class_id}))
print("CEP(release_d0, 1):", evaluator.evaluate(cep, {"x": release_d0, "k": pce_class_id}))
print("CEP(release_d1, 1):", evaluator.evaluate(cep, {"x": release_d1, "k": pce_class_id}))


CEP(release_s0, 1): 0.3499999212589552
CEP(release_s1, 1): 0.6499999051986434
CEP(release_d0, 1): 0.9000000508812466
CEP(release_d1, 1): 0.0


In [21]:
pce_delta = 0.8
pce = parser.parse(
    "FULL(x) * (y <<< x) * (CEP(y, k) >= delta)"
)

# Releasing D0 is a PCE; releasing D1 is not.
print("Release D0 is a PCE:", bool(evaluator.evaluate(
    pce,
    {"x": pce_instance, "y": release_d0, "k": pce_class_id, "delta": pce_delta},
)))
print("Release D1 is a PCE:", bool(evaluator.evaluate(
    pce,
    {"x": pce_instance, "y": release_d1, "k": pce_class_id, "delta": pce_delta},
)))

Release D0 is a PCE: True
Release D1 is a PCE: False


## MinimumPCE

`FREE(x, candidate)` counts how many features of `x` were released. A MinimumPCE satisfies the CEP threshold while releasing as few features as possible.

In [22]:
free = parser.parse("FREE(x, y)")

# Each candidate below releases exactly one feature.
print("Release S1:", evaluator.evaluate(free, {"x": pce_instance, "y": release_s1}))
print("Release D0:", evaluator.evaluate(free, {"x": pce_instance, "y": release_d0}))

Release S1: 1
Release D0: 1


In [23]:
parser.define_function(
    "SCE",
    parameters=["x", "candidate", "c", "delta", "release_max"],
    body=(
        "FULL(x) * (candidate <<< x) * "
        "(CEP(candidate, c) >= delta) * "
        "(1 <= FREE(x, candidate)) * "
        "(FREE(x, candidate) <= release_max)"
    ),
)

parser.define_function(
    "MINIMUM_PCE",
    parameters=["x", "candidate", "c", "delta", "release_max"],
    body=(
        "SCE(x, candidate, c, delta, release_max) * "
        "(SIGMA z. ("
        "    SCE(x, z, c, delta, release_max) * "
        "    NOT(FREE(x, candidate) <= FREE(x, z))"
        ") == 0)"
    ),
)

In [24]:
minimum_pce = parser.parse(
    "MINIMUM_PCE(x, y, c, delta, release_max)"
)

# Releasing D0 is a MinimumPCE at delta=0.8.
bool(evaluator.evaluate(
    minimum_pce,
    {
        "x": pce_instance,
        "y": release_d0,
        "c": pce_class_id,
        "delta": pce_delta,
        "release_max": 8,
    },
))

True

## Most probable contrastive explanation (MPCE)

At `delta=0.8`, releasing `D0` is the unique one-feature MinimumPCE and gives `CEP=0.90`. Releasing a noise feature as well does not improve `CEP`, so the larger candidate is not minimum.

In [25]:
mpce_expression = (
    "MINIMUM_PCE(x, y, c, delta, release_max) * "
    "(SIGMA w. ("
    "    MINIMUM_PCE(x, w, c, delta, release_max) * "
    "    NOT(CEP(w, c) <= CEP(y, c))"
    ") == 0)"
)
mpce = parser.parse(mpce_expression)

In [26]:
mpce_context = {
    "x": pce_instance,
    "c": pce_class_id,
    "delta": pce_delta,
    "release_max": 8,
}

# D0 is the MPCE. Releasing D0 and N0 is sufficient but not minimum.
print("Release D0 is an MPCE:", bool(evaluator.evaluate(
    mpce, {**mpce_context, "y": release_d0}
)))
print("Release D0 and N0 is an MPCE:", bool(evaluator.evaluate(
    mpce, {**mpce_context, "y": release_d0_and_n0}
)))

Release D0 is an MPCE: True
Release D0 and N0 is an MPCE: False


## All-completions explanations

`ALL_COMP_SR(x, k, delta)` requires every complete assignment compatible with `x` to satisfy `EP(completion, k) >= delta`. `ALL_COMP_CE` applies the same universal check to `CEP`.

With `S0=0` and `S1=1`, the MUX selects `D2`. Therefore, fixing `D2=1` guarantees class 1, fixing `D2=0` guarantees the other class, and leaving `D2` undefined guarantees neither class.

In [8]:
guarantees_class = (0, 1, None, None, 1, None, None, None)
guarantees_other_class = (0, 1, None, None, 0, None, None, None)
ambiguous_evidence = (0, 1, None, None, None, None, None, None)
global_delta = 0.9

In [9]:
parser.define_function(
    "ALL_COMP_SR",
    parameters=["candidate", "c", "delta"],
    body=(
        "SIGMA completion. ("
        "    COMP(candidate, completion) * "
        "    NOT(EP(completion, c) >= delta)"
        ") == 0"
    ),
)

parser.define_function(
    "ALL_COMP_CE",
    parameters=["candidate", "c", "delta"],
    body=(
        "SIGMA completion. ("
        "    COMP(candidate, completion) * "
        "    NOT(CEP(completion, c) >= delta)"
        ") == 0"
    ),
)

In [10]:
all_comp_sr = parser.parse("ALL_COMP_SR(x, k, delta)")
all_comp_ce = parser.parse("ALL_COMP_CE(x, k, delta)")

In [11]:
context = {"x": guarantees_class, "k": 1, "delta": global_delta}

# Show SR and CE. Expect True and False.
bool(evaluator.evaluate(all_comp_sr, context)), bool(evaluator.evaluate(all_comp_ce, context))

(True, False)

In [12]:
context = {"x": guarantees_other_class, "k": 1, "delta": global_delta}

# Show SR and CE. Expect False and True.
bool(evaluator.evaluate(all_comp_sr, context)), bool(evaluator.evaluate(all_comp_ce, context))

(False, True)

In [13]:
context = {"x": ambiguous_evidence, "k": 1, "delta": global_delta}

# Show SR and CE. Expect False and False.
bool(evaluator.evaluate(all_comp_sr, context)), bool(evaluator.evaluate(all_comp_ce, context))

(False, False)

## Threshold-determined assignments

A partial assignment is threshold-determined when all its completions support class `k`, or all its completions support the other class.

In [14]:
parser.define_function(
    "THRESHOLD_DETERMINED",
    parameters=["candidate", "c", "delta"],
    body=(
        "NOT("
        "    NOT(ALL_COMP_SR(candidate, c, delta)) * "
        "    NOT(ALL_COMP_CE(candidate, c, delta))"
        ")"
    ),
)

In [15]:
threshold_determined = parser.parse(
    "THRESHOLD_DETERMINED(x, c, delta)"
)

In [16]:
# Both assignments are determined, but in favor of different classes.
class_result = evaluator.evaluate(
    threshold_determined,
    {"x": guarantees_class, "c": 1, "delta": global_delta},
)
other_class_result = evaluator.evaluate(
    threshold_determined,
    {"x": guarantees_other_class, "c": 1, "delta": global_delta},
)

# Show both results. Expect True and True.
bool(class_result), bool(other_class_result)

(True, True)

In [17]:
# D2 is unknown, so different completions produce different classes.
bool(evaluator.evaluate(
    threshold_determined,
    {"x": ambiguous_evidence, "c": 1, "delta": global_delta},
))  # Expect False.

False

## Determinant feature sets (DFS)

A DFS is stronger than one threshold-determined assignment. It requires every assignment with the same defined positions to be threshold-determined. For this noisy MUX, all six functional features are required, but `N0` and `N1` are optional.

The following nested query is evaluated exactly by brute force, so it can take longer than the previous cells.

In [18]:
parser.define_function(
    "DFS",
    parameters=["candidate", "c", "delta"],
    body=(
        "SIGMA assignment. ("
        "    SUF(candidate, assignment) * "
        "    NOT(THRESHOLD_DETERMINED(assignment, c, delta))"
        ") == 0"
    ),
)

dfs = parser.parse("DFS(x, k, delta)")

In [19]:
partial_support = (0, 1, None, None, 1, None, None, None)

# This particular assignment determines class 1, but its support is not a DFS.
bool(evaluator.evaluate(
    dfs,
    {"x": partial_support, "k": 1, "delta": global_delta},
))  # Expect False.

False

In [20]:
core_support = (0, 1, 0, 1, 1, 0, None, None)
all_features_support = (0, 1, 0, 1, 1, 0, 0, 1)

# Noise is optional: both supports are determinant feature sets.
core_is_dfs = evaluator.evaluate(
    dfs,
    {"x": core_support, "k": 1, "delta": global_delta},
)
all_features_is_dfs = evaluator.evaluate(
    dfs,
    {"x": all_features_support, "k": 1, "delta": global_delta},
)

# Expect True and True.
bool(core_is_dfs), bool(all_features_is_dfs)

(True, True)

## Minimal determinant feature sets

A DFS is minimal when none of its strict subassignments is also a DFS. Here, `core_support` is the unique minimal DFS. Adding `N0`, `N1`, or both preserves the DFS property, but the resulting support is not minimal.

In [21]:
parser.define_function(
    "MINIMAL_DFS",
    parameters=["candidate", "c", "delta"],
    body=(
        "DFS(candidate, c, delta) * "
        "(SIGMA smaller. ("
        "    (smaller << candidate) * DFS(smaller, c, delta)"
        " ) == 0)"
    ),
)

minimal_dfs = parser.parse("MINIMAL_DFS(x, c, delta)")

Evaluating `minimal_dfs` directly would add a third nested exhaustive search. The exact test oracle checks all supports; here we inspect the simple witness manually.

In [22]:
strict_subset = parser.parse("smaller << candidate")
core_is_strict_subset = evaluator.evaluate(
    strict_subset,
    {"smaller": core_support, "candidate": all_features_support},
)

# Expect True, True, True.
# Since the smaller core is a DFS, all_features_support is not minimal.
bool(core_is_dfs), bool(all_features_is_dfs), bool(core_is_strict_subset)

(True, True, True)

In [23]:
# The complete query is available for experimentation, but it is slow:
# evaluator.evaluate(
#     minimal_dfs,
#     {"x": core_support, "c": 1, "delta": global_delta},
# )

## Distribution-aware DFS score

A DFS requires every assignment with the selected support to be threshold-determined. `DFS_SCORE` relaxes this requirement and measures the probability mass of the assignments that are threshold-determined.

In [24]:
parser.define_function(
    "DFS_SCORE",
    parameters=["candidate", "c", "delta"],
    body=(
        "SIGMA assignment. ("
        "    SUF(candidate, assignment) * "
        "    VAL(assignment) * "
        "    THRESHOLD_DETERMINED(assignment, c, delta)"
        ")"
    ),
)

dfs_score = parser.parse("DFS_SCORE(x, c, delta)")

In [25]:
support_d1_d2_d3 = (0, 1, None, 1, 1, 0, None, None)
support_d0_d2_d3 = (0, 1, 0, None, 1, 0, None, None)
support_d0_d1_d2 = (0, 1, 0, 1, 1, None, None, None)
smaller_support = (0, 1, 0, None, 1, None, None, None)

# Only the defined positions matter; their particular values do not.
score_d1_d2_d3 = evaluator.evaluate(
    dfs_score,
    {"x": support_d1_d2_d3, "c": 1, "delta": global_delta},
)
score_d0_d2_d3 = evaluator.evaluate(
    dfs_score,
    {"x": support_d0_d2_d3, "c": 1, "delta": global_delta},
)
score_d0_d1_d2 = evaluator.evaluate(
    dfs_score,
    {"x": support_d0_d1_d2, "c": 1, "delta": global_delta},
)
score_smaller = evaluator.evaluate(
    dfs_score,
    {"x": smaller_support, "c": 1, "delta": global_delta},
)

In [26]:
# Expect 0.7725, 0.8775, 0.7725, and 0.65.
score_d1_d2_d3, score_d0_d2_d3, score_d0_d1_d2, score_smaller

(0.7724999772365486,
 0.8774999019850718,
 0.7725000013192105,
 0.6499999638034003)

## Probabilistic determinant feature sets (ProbDFS)

A support is a `ProbDFS` when its `DFS_SCORE` reaches the mass threshold `rho`. We use `rho=0.75`, so the three five-feature supports qualify, while the four-feature support with score `0.65` does not.

In [27]:
parser.define_function(
    "PROB_DFS",
    parameters=["candidate", "c", "delta", "rho"],
    body="DFS_SCORE(candidate, c, delta) >= rho",
)

prob_dfs = parser.parse("PROB_DFS(x, c, delta, rho)")
prob_dfs_rho = 0.75

In [28]:
# These comparisons show the same decision made by PROB_DFS.
# Expect True, True, True, and False.
(
    score_d1_d2_d3 >= prob_dfs_rho,
    score_d0_d2_d3 >= prob_dfs_rho,
    score_d0_d1_d2 >= prob_dfs_rho,
    score_smaller >= prob_dfs_rho,
)

(True, True, True, False)

## MinimumProbDFS

`MinimumProbDFS` first minimizes the number of selected features. At `rho=0.75`, no support with four or fewer features qualifies. The three qualifying five-feature supports above are therefore all minimum.

In [29]:
parser.define_function(
    "MINIMUM_PROB_DFS",
    parameters=["candidate", "c", "delta", "rho"],
    body=(
        "PROB_DFS(candidate, c, delta, rho) * "
        "(SIGMA other. ("
        "    PROB_DFS(other, c, delta, rho) * "
        "    NOT(SIZE(candidate) <= SIZE(other))"
        ") == 0)"
    ),
)

minimum_prob_dfs = parser.parse(
    "MINIMUM_PROB_DFS(x, c, delta, rho)"
)

In [30]:
minimum_prob_dfs_candidates = (
    support_d1_d2_d3,
    support_d0_d2_d3,
    support_d0_d1_d2,
)

# Show that all three candidates select five features.
tuple(sum(value is not None for value in candidate)
      for candidate in minimum_prob_dfs_candidates)

(5, 5, 5)

## Most probable determinant feature set (MDFS)

MDFS breaks the tie between minimum-size `ProbDFS` candidates by maximizing `DFS_SCORE`. Therefore, the support `{S0, S1, D0, D2, D3}` wins with score `0.8775`.

In [3]:
parser.define_function(
    "MDFS",
    parameters=["candidate", "c", "delta", "rho"],
    body=(
        "MINIMUM_PROB_DFS(candidate, c, delta, rho) * "
        "(SIGMA other. ("
        "    MINIMUM_PROB_DFS(other, c, delta, rho) * "
        "    NOT(DFS_SCORE(other, c, delta) <= "
        "        DFS_SCORE(candidate, c, delta))"
        ") == 0)"
    ),
)

mdfs = parser.parse("MDFS(x, c, delta, rho)")

In [32]:
# The middle candidate has the strictly largest DFS score.
score_d0_d2_d3 > score_d1_d2_d3 and score_d0_d2_d3 > score_d0_d1_d2

True

In [ ]:
# The complete nested searches are available, but expensive:
# evaluator.evaluate(
#     minimum_prob_dfs,
#     {"x": support_d0_d2_d3, "c": 1,
#      "delta": global_delta, "rho": prob_dfs_rho},
# )

In [ ]:
# evaluator.evaluate(
#     mdfs,
#     {"x": support_d0_d2_d3, "c": 1, "delta": global_delta, "rho": prob_dfs_rho},
# )